# STG_6.2 — Comparación de modelos con features de autoría (solo 2019+)

Toma `data/df_entrenamiento_autor.csv` (spec 013 / STG_5.3) y compara los **mismos 6 modelos**
de STG_6 (spec 009), corriéndolos en **dos escenarios**:

- **Escenario A**: PRO fuera de la coalición de Milei (`es_oficialista`, `coincide_bloque_autor`)
- **Escenario B**: PRO dentro de la coalición de Milei (`es_oficialista_b`, `coincide_bloque_autor_b`)

Validación: `TimeSeriesSplit` + holdout temporal (idéntico esquema a STG_6). Métrica: **F1-macro**.

Se compara el mejor resultado contra el benchmark histórico de la spec 009 (F1-macro = 0.453),
aclarando que la población cambió (2019+ vs. todos los años).

In [1]:
import pandas as pd
import numpy as np
import joblib
import warnings
warnings.filterwarnings('ignore')
from pathlib import Path

DATA_DIR    = Path("../data")
SPEC_DIR    = Path("../specs/013-reentrenar-modelo-features-autor")
RANDOM_STATE = 42


## T11 — Carga, target y definición de features por escenario

In [2]:
# T11 — Cargar dataset, definir features y target
df = pd.read_csv(DATA_DIR / "df_entrenamiento_autor.csv", parse_dates=["fecha_votacion"])
df = df.sort_values("fecha_votacion").reset_index(drop=True)

META   = ["diputado", "titulo_base", "fecha_votacion", "id_votacion", "bloque", "provincia", "tema_label"]
TARGET = "voto"

# Features base (idénticas a STG_5/STG_6, spec 009) + features de autor comunes a A y B
FEATURES_COMUNES = [c for c in df.columns if c not in META + [TARGET]
                     and c not in ("es_oficialista", "coincide_bloque_autor",
                                    "es_oficialista_b", "coincide_bloque_autor_b")]

FEATURES_A = FEATURES_COMUNES + ["es_oficialista", "coincide_bloque_autor"]
FEATURES_B = FEATURES_COMUNES + ["es_oficialista_b", "coincide_bloque_autor_b"]

print(f"Filas              : {len(df):,}")
print(f"Features comunes   : {len(FEATURES_COMUNES)} (base STG_5 + bloque_autor_enc + es_poder_ejecutivo)")
print(f"Features escenario A: {len(FEATURES_A)} (+ es_oficialista, coincide_bloque_autor)")
print(f"Features escenario B: {len(FEATURES_B)} (+ es_oficialista_b, coincide_bloque_autor_b)")
print(f"Target             : {TARGET}")
print()

# Codificar el target con LabelEncoder y guardarlo (nombre propio, sin pisar el de la spec 009)
from sklearn.preprocessing import LabelEncoder
le_voto = LabelEncoder()
df["voto_enc"] = le_voto.fit_transform(df[TARGET])
joblib.dump(le_voto, DATA_DIR / "le_voto_autor.joblib")

print("Clases del target:")
for i, cls in enumerate(le_voto.classes_):
    n = (df["voto_enc"] == i).sum()
    print(f"  {i} = {cls}: {n:,} ({n/len(df)*100:.1f}%)")


Filas              : 20,608
Features comunes   : 396 (base STG_5 + bloque_autor_enc + es_poder_ejecutivo)
Features escenario A: 398 (+ es_oficialista, coincide_bloque_autor)
Features escenario B: 398 (+ es_oficialista_b, coincide_bloque_autor_b)
Target             : voto



Clases del target:
  0 = ABSTENCIÓN: 679 (3.3%)
  1 = AFIRMATIVO: 15,781 (76.6%)
  2 = NEGATIVO: 4,148 (20.1%)


In [3]:
# Verificaciones T11
assert len(df) == 20608, f"Se esperaban 20.608 filas, hay {len(df)}"
assert df[FEATURES_A].isna().sum().sum() == 0, "Hay NaN en features del escenario A"
assert df[FEATURES_B].isna().sum().sum() == 0, "Hay NaN en features del escenario B"
assert len(FEATURES_A) == len(FEATURES_B), "Los dos escenarios deben tener la misma cantidad de features"
assert set(FEATURES_A) - set(FEATURES_B) == {"es_oficialista", "coincide_bloque_autor"}
assert set(FEATURES_B) - set(FEATURES_A) == {"es_oficialista_b", "coincide_bloque_autor_b"}
assert df["fecha_votacion"].min() >= pd.Timestamp("2019-01-01")

print()
print("✓ T11 PASA: dataset cargado, target codificado, features A/B definidas correctamente.")



✓ T11 PASA: dataset cargado, target codificado, features A/B definidas correctamente.


## T12 — Split temporal y TimeSeriesSplit

In [4]:
from sklearn.model_selection import TimeSeriesSplit

# Partición pasado/futuro: el 80% más antiguo entrena, el 20% más reciente es el holdout
corte_idx = int(len(df) * 0.80)
fecha_corte = df.iloc[corte_idx]["fecha_votacion"]

df_train = df.iloc[:corte_idx].copy()
df_hold  = df.iloc[corte_idx:].copy()

y_train = df_train["voto_enc"].values
y_hold  = df_hold["voto_enc"].values

# Arrays X por escenario (mismo split de filas, distintas columnas de features)
X_train_A = df_train[FEATURES_A].values
X_hold_A  = df_hold[FEATURES_A].values
X_train_B = df_train[FEATURES_B].values
X_hold_B  = df_hold[FEATURES_B].values

tscv = TimeSeriesSplit(n_splits=5)

print(f"Fecha de corte train/holdout : {fecha_corte.date()}")
print(f"Train : {len(df_train):,} filas ({df_train['fecha_votacion'].min().date()} a {df_train['fecha_votacion'].max().date()})")
print(f"Holdout: {len(df_hold):,} filas ({df_hold['fecha_votacion'].min().date()} a {df_hold['fecha_votacion'].max().date()})")
print()
print("Distribución de clases en holdout:")
for i, cls in enumerate(le_voto.classes_):
    n = (y_hold == i).sum()
    print(f"  {cls}: {n:,} ({n/len(y_hold)*100:.1f}%)")


Fecha de corte train/holdout : 2025-12-18
Train : 16,486 filas (2019-04-04 a 2025-12-18)
Holdout: 4,122 filas (2025-12-18 a 2026-05-20)

Distribución de clases en holdout:
  ABSTENCIÓN: 44 (1.1%)
  AFIRMATIVO: 3,459 (83.9%)
  NEGATIVO: 619 (15.0%)


In [5]:
# Verificación: ninguna fecha del train supera la mínima del holdout
assert df_train["fecha_votacion"].max() <= df_hold["fecha_votacion"].min(),     "VIOLATION: hay fechas del train posteriores al holdout"

# Verificar que TimeSeriesSplit respeta orden: cada fold valida con datos mas nuevos que el train
for fold, (tr_idx, val_idx) in enumerate(tscv.split(X_train_A)):
    max_tr  = df_train.iloc[tr_idx]["fecha_votacion"].max()
    min_val = df_train.iloc[val_idx]["fecha_votacion"].min()
    assert max_tr <= min_val, f"Fold {fold}: fechas del train posteriores a la validación"

print(f"TimeSeriesSplit con {tscv.n_splits} folds verificado: orden temporal correcto en todos los folds.")
print("✓ T12 PASA.")


TimeSeriesSplit con 5 folds verificado: orden temporal correcto en todos los folds.
✓ T12 PASA.


## T13 — Comparación de 6 modelos: Escenario A (PRO fuera de LLA)

In [6]:
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, BaggingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import cross_val_score
from sklearn.metrics import f1_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

def evaluar_modelo(nombre, modelo, X_tr, y_tr, X_ho, y_ho, tscv, cv_n_jobs=-1):
    """Evalua un modelo con CV temporal y en el holdout. Devuelve dict con metricas."""
    scores_cv = cross_val_score(modelo, X_tr, y_tr, cv=tscv, scoring='f1_macro', n_jobs=cv_n_jobs)
    modelo.fit(X_tr, y_tr)
    y_pred_hold = modelo.predict(X_ho)
    f1_hold = f1_score(y_ho, y_pred_hold, average='macro')
    print(f"  {nombre:<30} CV F1-macro: {scores_cv.mean():.3f} ± {scores_cv.std():.3f} | Holdout: {f1_hold:.3f}")
    return {'modelo': nombre, 'cv_mean': scores_cv.mean(), 'cv_std': scores_cv.std(), 'holdout': f1_hold, 'estimator': modelo}


def correr_6_modelos(X_train, y_train, X_hold, y_hold, tscv, etiqueta):
    """Corre los mismos 6 modelos de STG_6 (spec 009) y devuelve el dict de resultados."""
    resultados = {}

    dummy = DummyClassifier(strategy='most_frequent', random_state=RANDOM_STATE)
    resultados['Dummy'] = evaluar_modelo('Dummy (most_frequent)', dummy, X_train, y_train, X_hold, y_hold, tscv)

    logreg_pipe = Pipeline([
        ('scaler', StandardScaler()),
        ('clf', LogisticRegression(max_iter=1000, random_state=RANDOM_STATE, class_weight='balanced'))
    ])
    resultados['LogReg'] = evaluar_modelo('LogisticRegression', logreg_pipe, X_train, y_train, X_hold, y_hold, tscv)

    rf = RandomForestClassifier(n_estimators=200, class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1)
    resultados['RF'] = evaluar_modelo('RandomForest', rf, X_train, y_train, X_hold, y_hold, tscv)

    bag = BaggingClassifier(estimator=DecisionTreeClassifier(class_weight='balanced'),
                            n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1)
    resultados['Bagging'] = evaluar_modelo('BaggingClassifier', bag, X_train, y_train, X_hold, y_hold, tscv)

    xgb = XGBClassifier(
        n_estimators=300, learning_rate=0.05, max_depth=6,
        eval_metric='mlogloss', random_state=RANDOM_STATE, n_jobs=-1, verbosity=0
    )
    resultados['XGB'] = evaluar_modelo('XGBClassifier', xgb, X_train, y_train, X_hold, y_hold, tscv)

    # LightGBM: n_jobs=1 en cross_val_score para evitar doble paralelismo (bug documentado, spec 009)
    lgbm = LGBMClassifier(
        n_estimators=300, learning_rate=0.05, num_leaves=63,
        class_weight='balanced', random_state=RANDOM_STATE, n_jobs=-1, verbosity=-1
    )
    resultados['LGBM'] = evaluar_modelo('LGBMClassifier', lgbm, X_train, y_train, X_hold, y_hold, tscv, cv_n_jobs=1)

    return resultados


print(f"Entrenando 6 modelos — Escenario A ({len(FEATURES_A)} features)...")
print()
resultados_A = correr_6_modelos(X_train_A, y_train, X_hold_A, y_hold, tscv, 'A')
print()
print("✓ T13 PASA: 6 modelos entrenados y evaluados en escenario A.")


Entrenando 6 modelos — Escenario A (398 features)...



  Dummy (most_frequent)          CV F1-macro: 0.279 ± 0.019 | Holdout: 0.304


  LogisticRegression             CV F1-macro: 0.408 ± 0.026 | Holdout: 0.108


  RandomForest                   CV F1-macro: 0.348 ± 0.069 | Holdout: 0.412


  BaggingClassifier              CV F1-macro: 0.385 ± 0.094 | Holdout: 0.526


  XGBClassifier                  CV F1-macro: 0.403 ± 0.060 | Holdout: 0.489


  LGBMClassifier                 CV F1-macro: 0.406 ± 0.074 | Holdout: 0.581

✓ T13 PASA: 6 modelos entrenados y evaluados en escenario A.


## T14 — Comparación de 6 modelos: Escenario B (PRO dentro de LLA)

In [7]:
print(f"Entrenando 6 modelos — Escenario B ({len(FEATURES_B)} features)...")
print()
resultados_B = correr_6_modelos(X_train_B, y_train, X_hold_B, y_hold, tscv, 'B')
print()
print("✓ T14 PASA: 6 modelos entrenados y evaluados en escenario B.")


Entrenando 6 modelos — Escenario B (398 features)...



  Dummy (most_frequent)          CV F1-macro: 0.279 ± 0.019 | Holdout: 0.304


  LogisticRegression             CV F1-macro: 0.407 ± 0.038 | Holdout: 0.126


  RandomForest                   CV F1-macro: 0.349 ± 0.070 | Holdout: 0.402


  BaggingClassifier              CV F1-macro: 0.371 ± 0.098 | Holdout: 0.488


  XGBClassifier                  CV F1-macro: 0.401 ± 0.066 | Holdout: 0.527


  LGBMClassifier                 CV F1-macro: 0.409 ± 0.081 | Holdout: 0.531

✓ T14 PASA: 6 modelos entrenados y evaluados en escenario B.


## T15 — Tabla comparativa (12 filas) y elección del ganador

In [8]:
filas = []
for escenario, resultados in [('A (PRO fuera de LLA)', resultados_A), ('B (PRO dentro de LLA)', resultados_B)]:
    for v in resultados.values():
        filas.append({
            'Modelo': v['modelo'],
            'Escenario': escenario,
            'CV F1-macro (media)': round(v['cv_mean'], 3),
            'CV F1-macro (desvío)': round(v['cv_std'], 3),
            'Holdout F1-macro': round(v['holdout'], 3),
        })

tabla = pd.DataFrame(filas).sort_values('Holdout F1-macro', ascending=False).reset_index(drop=True)

print("=" * 80)
print("TABLA COMPARATIVA — 6 modelos x 2 escenarios (12 filas)")
print("=" * 80)
print(tabla.to_string(index=False))
print("=" * 80)

tabla.to_csv(SPEC_DIR / "tabla_comparativa_modelos_autor.csv", index=False)
print()
print("Tabla guardada en specs/013-reentrenar-modelo-features-autor/tabla_comparativa_modelos_autor.csv")

# Elegir ganador por F1-macro en holdout
fila_ganadora = tabla.iloc[0]
nombre_ganador = fila_ganadora['Modelo']
escenario_ganador = fila_ganadora['Escenario']
resultados_ganador = resultados_A if escenario_ganador.startswith('A') else resultados_B
ganador = next(v['estimator'] for v in resultados_ganador.values() if v['modelo'] == nombre_ganador)
X_hold_ganador = X_hold_A if escenario_ganador.startswith('A') else X_hold_B
FEATURES_ganador = FEATURES_A if escenario_ganador.startswith('A') else FEATURES_B

print()
print(f"GANADOR: {nombre_ganador} — Escenario {escenario_ganador}  (Holdout F1-macro = {fila_ganadora['Holdout F1-macro']})")
print("✓ T15a PASA.")


TABLA COMPARATIVA — 6 modelos x 2 escenarios (12 filas)
               Modelo             Escenario  CV F1-macro (media)  CV F1-macro (desvío)  Holdout F1-macro
       LGBMClassifier  A (PRO fuera de LLA)                0.406                 0.074             0.581
       LGBMClassifier B (PRO dentro de LLA)                0.409                 0.081             0.531
        XGBClassifier B (PRO dentro de LLA)                0.401                 0.066             0.527
    BaggingClassifier  A (PRO fuera de LLA)                0.385                 0.094             0.526
        XGBClassifier  A (PRO fuera de LLA)                0.403                 0.060             0.489
    BaggingClassifier B (PRO dentro de LLA)                0.371                 0.098             0.488
         RandomForest  A (PRO fuera de LLA)                0.348                 0.069             0.412
         RandomForest B (PRO dentro de LLA)                0.349                 0.070             0.402

In [9]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

# --- Gráfico 1: comparación de las 12 combinaciones (barras horizontales) ---
tabla_plot = tabla.sort_values('Holdout F1-macro')
etiquetas = tabla_plot['Modelo'] + ' — ' + tabla_plot['Escenario'].str.slice(0, 1)

fig, ax = plt.subplots(figsize=(10, 7))
y = np.arange(len(tabla_plot))
h = 0.35
ax.barh(y + h/2, tabla_plot['CV F1-macro (media)'], h, color='#5B9BD5', label='CV F1-macro (media)')
ax.barh(y - h/2, tabla_plot['Holdout F1-macro'],     h, color='#ED7D31', label='Holdout F1-macro')
ax.set_yticks(y)
ax.set_yticklabels(etiquetas, fontsize=9)
ax.set_xlabel('F1-macro')
ax.set_title('Comparación de modelos x escenario (A/B) — F1-macro', fontsize=13, fontweight='bold')
ax.axvline(x=0.453, color='green', linestyle='--', linewidth=1.2, label='Benchmark histórico (spec 009) = 0.453')
ax.legend(loc='lower right', fontsize=8)
ax.set_xlim(0, 0.65)
for bar in ax.patches:
    w = bar.get_width()
    ax.text(w + 0.005, bar.get_y() + bar.get_height()/2, f'{w:.3f}', va='center', fontsize=7)
plt.tight_layout()
plt.savefig(SPEC_DIR / 'comparacion_modelos_autor.png', dpi=150, bbox_inches='tight')
plt.close()
print("Guardado: comparacion_modelos_autor.png")

# --- Gráfico 2: matriz de confusión del ganador en holdout ---
y_pred_hold = ganador.predict(X_hold_ganador)
cm = confusion_matrix(y_hold, y_pred_hold)
clases = le_voto.classes_

fig, ax = plt.subplots(figsize=(6, 5))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=clases)
disp.plot(ax=ax, colorbar=False, cmap='Blues')
ax.set_title(f'Matriz de confusión — {nombre_ganador} (Escenario {escenario_ganador})\n'
            f'(holdout: {df_hold["fecha_votacion"].min().date()} a {df_hold["fecha_votacion"].max().date()})',
             fontsize=10, fontweight='bold')
plt.tight_layout()
plt.savefig(SPEC_DIR / 'matriz_confusion_ganador_autor.png', dpi=150, bbox_inches='tight')
plt.close()
print("Guardado: matriz_confusion_ganador_autor.png")

print()
print("Matriz de confusión (filas=real, columnas=predicho):")
print(f"  Clases: {list(clases)}")
for i, fila in enumerate(cm):
    print(f"  {clases[i]:<12}: {list(fila)}")

print()
print("✓ T15 PASA: tabla comparativa, ganador y gráficos guardados.")


Guardado: comparacion_modelos_autor.png


Guardado: matriz_confusion_ganador_autor.png

Matriz de confusión (filas=real, columnas=predicho):
  Clases: ['ABSTENCIÓN', 'AFIRMATIVO', 'NEGATIVO']
  ABSTENCIÓN  : [np.int64(4), np.int64(23), np.int64(17)]
  AFIRMATIVO  : [np.int64(1), np.int64(3383), np.int64(75)]
  NEGATIVO    : [np.int64(2), np.int64(283), np.int64(334)]

✓ T15 PASA: tabla comparativa, ganador y gráficos guardados.


## T16 — Comparación contra el benchmark histórico (0.453) y resolución A vs. B

**Aclaración de contexto (no es una comparación pareja):** el benchmark 0.453 (spec 009) se
calculó entrenando con **todos los años** (1993-2026). Este modelo entrena solo con **2019 en
adelante**, con menos datos y otra composición de la Cámara. Una diferencia puede deberse al
cambio de población, no solo a las features de autoría.

In [10]:
BENCHMARK_HISTORICO = 0.453

f1_ganador = fila_ganadora['Holdout F1-macro']
mejora_pp = f1_ganador - BENCHMARK_HISTORICO
mejora_pct = mejora_pp / BENCHMARK_HISTORICO * 100

print("=" * 70)
print("COMPARACIÓN CONTRA EL BENCHMARK HISTÓRICO (spec 009)")
print("=" * 70)
print(f"Benchmark histórico (todos los años)      : {BENCHMARK_HISTORICO:.3f}")
print(f"Ganador (2019+, con autoría) — {nombre_ganador} / Esc. {escenario_ganador[0]}: {f1_ganador:.3f}")
print(f"Diferencia                                : {mejora_pp:+.3f} ({mejora_pct:+.1f}%)")
print()
print("NOTA: comparación de referencia, no pareja (distinta población: 2019+ vs. todos los años).")

# --- Resolución A vs B ---
fila_A = tabla[(tabla['Modelo'] == 'LGBMClassifier') & (tabla['Escenario'].str.startswith('A'))].iloc[0]
fila_B = tabla[(tabla['Modelo'] == 'LGBMClassifier') & (tabla['Escenario'].str.startswith('B'))].iloc[0]
diferencia_AB = abs(fila_A['Holdout F1-macro'] - fila_B['Holdout F1-macro'])
cv_std_max = max(fila_A['CV F1-macro (desvío)'], fila_B['CV F1-macro (desvío)'])

print()
print("=" * 70)
print("RESOLUCIÓN A vs. B (PRO fuera / dentro de la coalición de Milei)")
print("=" * 70)
print(f"LGBM Escenario A — Holdout: {fila_A['Holdout F1-macro']:.3f} | CV std: {fila_A['CV F1-macro (desvío)']:.3f}")
print(f"LGBM Escenario B — Holdout: {fila_B['Holdout F1-macro']:.3f} | CV std: {fila_B['CV F1-macro (desvío)']:.3f}")
print(f"Diferencia de holdout entre A y B: {diferencia_AB:.3f}")
print(f"Mayor desvío de CV entre ambos   : {cv_std_max:.3f}")
print()
if diferencia_AB < cv_std_max:
    print(f"La diferencia ({diferencia_AB:.3f}) es MENOR al desvío de la CV ({cv_std_max:.3f}):")
    print("estadísticamente NO CONCLUYENTE por sí sola. Se requiere un criterio secundario")
    print("explícito para decidir la definición oficial (ver memoria/DECISIONES.md).")
else:
    print(f"La diferencia ({diferencia_AB:.3f}) es MAYOR al desvío de la CV ({cv_std_max:.3f}):")
    print("hay evidencia de que un escenario domina al otro.")

print()
print("✓ T16 PASA.")


COMPARACIÓN CONTRA EL BENCHMARK HISTÓRICO (spec 009)
Benchmark histórico (todos los años)      : 0.453
Ganador (2019+, con autoría) — LGBMClassifier / Esc. A: 0.581
Diferencia                                : +0.128 (+28.3%)

NOTA: comparación de referencia, no pareja (distinta población: 2019+ vs. todos los años).

RESOLUCIÓN A vs. B (PRO fuera / dentro de la coalición de Milei)
LGBM Escenario A — Holdout: 0.581 | CV std: 0.074
LGBM Escenario B — Holdout: 0.531 | CV std: 0.081
Diferencia de holdout entre A y B: 0.050
Mayor desvío de CV entre ambos   : 0.081

La diferencia (0.050) es MENOR al desvío de la CV (0.081):
estadísticamente NO CONCLUYENTE por sí sola. Se requiere un criterio secundario
explícito para decidir la definición oficial (ver memoria/DECISIONES.md).

✓ T16 PASA.
